# Setup

## Libraries

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from PIL import Image
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import optuna
import gc
import struct
import seaborn as sns
import itertools
import copy
from sklearn.datasets import load_digits

import sys
sys.path.append("C:/Users/har23/Desktop/University/Lab/TCAM_Python/Code/utils_07")
from CTCAM_utils import *
from EVALUATION_utils import *
from FILTER_utils import *
from LAYER_utils import *
from MODEL_utils import *
from SAVEnLOAD_utils import *
from DATA_utils import *

gc.enable()


## User define

In [ ]:
random_seed = 42

## save options
train_save_options = {"save_image_indices": [], 
                      "save_image_dir": "./../image_save_temp", 
                      "save_image_type": ".txt", 
                      "save_image_one_file": True, 
                      "save_image_float": False, 
                      "save_image_hex": False}
test_save_options = {"save_image_indices": [], 
                      "save_image_dir": "./../image_save_temp", 
                      "save_image_type": ".txt", 
                      "save_image_one_file": True, 
                      "save_image_float": False, 
                      "save_image_hex": True, 
                      "save_score_indices": [], 
                      "save_score_dir": "./../score_save_temp"}
save_count = False # save count of trained model
save_count_dir = "./../count_save_temp"
save_count_sep = False 

## data
data_use = "digits" # # MNIST original dataset: "mnist_original"(train 60000, test 10000), MNIST: "mnist", DIGITS: "digits", inside: "inside", outside: "outside"
train_size = 1500 # count or ratio
test_size = 297 # count or ratio
data_dir = "C:/Users/har23/Desktop/University/Lab/TCAM_Python/Data"

## tuning
optuna_log_name = "log_filename"
iter = 100 # optuna iteration
Random, TPE, Grid = [False, True, False] # Random: RandomSample based tuning, TPE: TPESampler based tuning, Grid: User defined search space(below) based sequential tuning
search_space = {'hyperparameter1': [1, 2, 3], 
                'hyperparameter2': [10, 20, 30], 
                'hyperparameter3': [0.1, 1., 10.]
}

## Random seed

In [ ]:
# set random seed for reproducibility
def seed_everything(seed):
    # random.seed(seed)  
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    # tf.random.set_seed(seed)
    # torch.manual_seed(seed)
    # torch.cuda.manual_seed(seed)
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = True

seed_everything(random_seed)

## Data type

In [ ]:
if (data_use == "inside"):
    label_num = 3 
    class_mapping = ["normal", "object_search", "sleepy"]

elif (data_use == "outside"):
    label_num = 4
    class_mapping = ["abnormal", "left_lane", "normal", "right_lane"]

elif (data_use == "mnist" or data_use == "mnist_original"):
    label_num = 10 
    class_mapping = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

elif (data_use == "digits"):
    label_num = 10 
    class_mapping = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
else:
    label_num=0
    raise()

# Run

## Data selection

In [ ]:
if (data_use=='inside'):
    X_train, X_test, y_train, y_test = data_selection_inout(data_dir=data_dir, data_use=data_use, class_mapping=class_mapping)
elif (data_use=='outside'):
    X_train, X_test, y_train, y_test = data_selection_inout(data_dir=data_dir, data_use=data_use, class_mapping=class_mapping)
elif (data_use=='mnist'):
    X_train, X_test, y_train, y_test = data_selection_mnist(data_dir=data_dir, train_size=train_size, test_size=test_size, random_seed=random_seed)
elif (data_use=='digits'):
    X_train, X_test, y_train, y_test = data_selection_digits(train_size=train_size, test_size=test_size, random_seed=random_seed)
elif (data_use=='mnist_original'):
    X_train, X_test, y_train, y_test = load_all_original_MNIST(data_dir=data_dir)
else:
    pass

## One Test

### Model define

In [ ]:
# Top 32
test_param = {}

eval_thresholds = []
score_vals = []
my_model = Model02(label_num=label_num, input_shape=(8,8))
my_model.add(Top_JustConv2DLayer(label_num=label_num, 
                                kernel_size=(2,2), 
                                strides=(1,1), 
                                use_weight_based_threshold=False, 
                                input_max=16, 
                                all_ctcam_creation=True, 
                                top_num=12, 
                                top_thresholds=[], 
                                filter_weight_min=1, 
                                dynamic_kernel_order=False))
my_model.add(NormalizationLayer(stretching=False, 
                                new_max=15, 
                                old_min=0, 
                                old_max=0, 
                                normalize_per_group=False, 
                                normalize_per_image=True, 
                                adaptability='Weak'))
my_model.add(ScoringLayer(strides=(1, 1), 
                            exclude_zero=False, 
                            nearn=False, 
                            use_weight_based_threshold=False, 
                            quant_thresholds=[], 
                            penalty_interval=0, 
                            thresholds=eval_thresholds, 
                            score_vals=score_vals))

### Fit

In [ ]:
################################################## Train ##############################################
my_model.fit(x=X_train, y=y_train, save_options=train_save_options)

if (save_count):
    save_top_filter_count(model=my_model, save_dir=save_count_dir, one_file=True)
    save_trained_count(model=my_model, save_dir=save_count_dir, one_file=True)
else:
    pass

### Prediction

In [ ]:
###################################################### Test ############################################################
scores, scores_each, every_scores = my_model.predict(x=X_test, save_options=test_save_options)
prediction = most_similar_group(scores=scores)
probabilities = score_proba(scores=scores)

#prediction_proba = most_similar_group_proba(scores=scores)

### Evaluation

In [ ]:
##### accuracy
print("--------------- accuracy ---------------")
accuracy = calc_accuracy(test_labels=y_test, most_similar_group=prediction)

##### confusion matrix
print("--------------- confusion matrix ---------------")
#show_confusion_matrix(label_num=label_num, test_labels=y_test, most_similar_group=prediction, axis=0, show_count=False)
show_cf_matrix_heatmap(label_num=label_num, test_labels=y_test, most_similar_group=prediction, axis=0, show_count=False)

##### precision, recall, f1 score, accuracy
print("--------------- precision, recall, f1 score, accuracy ---------------")
clf_report(y_test=y_test, prediction=prediction, class_mapping=class_mapping)

##### auc
print("--------------- AUC ---------------")
calc_auc(y_test=y_test, probabilities=probabilities)

##### roc curve
print("--------------- ROC curve ---------------")
plot_roc_curve(y_test=y_test, label_num=label_num, probabilities=probabilities, class_mapping=class_mapping)

# Tuning

## Optuna objective function

In [ ]:
def objective(trial):

    gc.collect()
    param = {
        'top_2quant_threshold': trial.suggest_int('top_2quant_threshold', 1, 15), 
        'score_2quant_threshold' : trial.suggest_int('score_2quant_threshold', 1, 15), 
        'eval_threshold_gap0': trial.suggest_int('eval_threshold_gap0', 1, 20, step=1), 
        'eval_threshold_gap1': trial.suggest_int('eval_threshold_gap1', 1, 20, step=1), 
        'eval_threshold_gap2': trial.suggest_int('eval_threshold_gap2', 1, 20, step=1), 
        'eval_threshold_gap3': trial.suggest_int('eval_threshold_gap3', 1, 20, step=1), 
        'eval_threshold_gap4': trial.suggest_int('eval_threshold_gap4', 1, 20, step=1),
        'eval_threshold_gap5': trial.suggest_int('eval_threshold_gap5', 1, 20, step=1),
        'score_val0': trial.suggest_int('score_val0', 1, 100, step=1),
        'score_val1': trial.suggest_int('score_val1', 1, 100, step=1),
        'score_val2': trial.suggest_int('score_val2', 1, 100, step=1),
        'score_val3': trial.suggest_int('score_val3', 1, 100, step=1),
        'score_val4': trial.suggest_int('score_val4', 1, 100, step=1),
        'score_val5': trial.suggest_int('score_val5', 1, 100, step=1),
    }
    eval_thresholds = []
    score_vals = []
    
    my_model = Model02(label_num=label_num, input_shape=(8,8))
    
    my_model.add(Top_JustConv2DLayer(label_num=label_num, 
                                    kernel_size=(2,2), 
                                    strides=(1,1), 
                                    use_weight_based_threshold=False, 
                                    input_max=16, 
                                    all_ctcam_creation=True, 
                                    top_num=12, 
                                    top_thresholds=[], 
                                    filter_weight_min=1, 
                                    dynamic_kernel_order=False))
    my_model.add(NormalizationLayer(stretching=False, 
                                    new_max=15, 
                                    old_min=0, 
                                    old_max=0, 
                                    normalize_per_group=False, 
                                    normalize_per_image=True, 
                                    adaptability='Weak'))
    my_model.add(ScoringLayer(strides=(1, 1), 
                                exclude_zero=False, 
                                nearn=False, 
                                use_weight_based_threshold=False, 
                                quant_thresholds=[], 
                                penalty_interval=0, 
                                thresholds=eval_thresholds, 
                                score_vals=score_vals))

    
    my_model.fit(x=X_train, y=y_train, save_options=train_save_options)

    scores, scores_each, every_scores = my_model.predict(x=X_test, save_options=test_save_options)
    prediction = most_similar_group(scores=scores)
    accuracy = calc_accuracy(test_labels=y_test, most_similar_group=prediction)


    return accuracy

## Study and optimize function

In [ ]:
def Start_optuna_RandomSampler(optuna_log_name, iter, random_seed):
    study_name = optuna_log_name
    storage_name = "sqlite:///{}.db".format(study_name)

    study = optuna.create_study(
       sampler=optuna.samplers.RandomSampler(seed=random_seed),
       direction='maximize',
       study_name=study_name,
       storage=storage_name,
       load_if_exists=True)

    study.optimize(objective, n_trials=iter)


def Start_optuna_TPESampler(optuna_log_name, iter, random_seed):
    study_name = optuna_log_name
    storage_name = "sqlite:///{}.db".format(study_name)

    study = optuna.create_study(
       sampler=optuna.samplers.TPESampler(seed=random_seed),
       direction='maximize',
       study_name=study_name,
       storage=storage_name,
       load_if_exists=True)

    study.optimize(objective, n_trials=iter)


def Start_optuna_GridSampler(optuna_log_name, iter, search_space, random_seed):
    study_name = optuna_log_name
    storage_name = "sqlite:///{}.db".format(study_name)

    study = optuna.create_study(
       sampler=optuna.samplers.GridSampler(search_space, seed=random_seed),
       direction='maximize',
       study_name=study_name,
       storage=storage_name,
       load_if_exists=True)

    study.optimize(objective, n_trials=iter)

## Start tuning

In [ ]:
if (Random):
    Start_optuna_RandomSampler(optuna_log_name=optuna_log_name, iter=iter, random_seed=random_seed)

elif (TPE):
    Start_optuna_TPESampler(optuna_log_name=optuna_log_name, iter=iter, random_seed=random_seed)

elif (Grid):
    Start_optuna_GridSampler(optuna_log_name=optuna_log_name, iter=iter, search_space=search_space, random_seed=random_seed)

else:
    pass

# Tail tuning

## Set tuning options

In [ ]:
optuna_log_name = "log_filename_tail"
iter = 500 # optuna iteration
Random, TPE, Grid = [False, True, False] # Random: RandomSample based tuning, TPE: TPESampler based tuning, Grid: User defined search space(below) based sequential tuning
search_space = {'hyperparameter1': [1, 2, 3], 
                'hyperparameter2': [10, 20, 30], 
                'hyperparameter3': [0.1, 1., 10.]
}

## Head model define

In [ ]:
test_param = {}

In [ ]:
my_model = Model02(label_num=label_num, input_shape=(8,8))
my_model.add(Top_JustConv2DLayer(label_num=label_num, 
                                kernel_size=(2,2), 
                                strides=(1,1), 
                                use_weight_based_threshold=False, 
                                input_max=16, 
                                all_ctcam_creation=True, 
                                top_num=12, 
                                top_thresholds=[], 
                                filter_weight_min=1, 
                                dynamic_kernel_order=False))
my_model.add(NormalizationLayer(stretching=False, 
                                new_max=15, 
                                old_min=0, 
                                old_max=0, 
                                normalize_per_group=False, 
                                normalize_per_image=True, 
                                adaptability='Weak'))
my_model.add(ScoringLayer(strides=(1, 1), 
                            exclude_zero=False, 
                            nearn=False, 
                            use_weight_based_threshold=False, 
                            quant_thresholds=[], 
                            penalty_interval=0, 
                            thresholds=[], 
                            score_vals=[]))

my_model.fit(x=X_train, y=y_train, save_options=train_save_options)
patterns_arr_set = my_model.get_intermediate_output(x=X_test)

## Optuna objective function for tail model

In [ ]:
def objective(trial):

    gc.collect()
    param = {
        'eval_threshold_gap0': trial.suggest_int('eval_threshold_gap0', 1, 20, step=1), 
        'eval_threshold_gap1': trial.suggest_int('eval_threshold_gap1', 1, 20, step=1), 
        'eval_threshold_gap2': trial.suggest_int('eval_threshold_gap2', 1, 20, step=1), 
        'eval_threshold_gap3': trial.suggest_int('eval_threshold_gap3', 1, 20, step=1), 
        'eval_threshold_gap4': trial.suggest_int('eval_threshold_gap4', 1, 20, step=1),
        'eval_threshold_gap5': trial.suggest_int('eval_threshold_gap5', 1, 20, step=1),
        'score_val0': trial.suggest_int('score_val0', 1, 100, step=1),
        'score_val1': trial.suggest_int('score_val1', 1, 100, step=1),
        'score_val2': trial.suggest_int('score_val2', 1, 100, step=1),
        'score_val3': trial.suggest_int('score_val3', 1, 100, step=1),
        'score_val4': trial.suggest_int('score_val4', 1, 100, step=1),
        'score_val5': trial.suggest_int('score_val5', 1, 100, step=1),
    }
    eval_thresholds = []
    score_vals = []
    
    
    my_model.layers[-1].thresholds = eval_thresholds
    my_model.layers[-1].score_vals = score_vals
    scores, scores_each, every_scores = my_model.finalize_prediction(patterns_arr_set=patterns_arr_set)
    prediction = most_similar_group(scores=scores)
    accuracy = calc_accuracy(test_labels=y_test, most_similar_group=prediction)

    return accuracy

## Start tuning

In [ ]:
if (Random):
    Start_optuna_RandomSampler(optuna_log_name=optuna_log_name, iter=iter, random_seed=random_seed)

elif (TPE):
    Start_optuna_TPESampler(optuna_log_name=optuna_log_name, iter=iter, random_seed=random_seed)

elif (Grid):
    Start_optuna_GridSampler(optuna_log_name=optuna_log_name, iter=iter, search_space=search_space, random_seed=random_seed)

else:
    pass